# SESSION 1 - Introduction to RAG & Why It's Needed

This notebook covers the fundamentals of Retrieval-Augmented Generation (RAG), including why modern AI applications require external knowledge retrieval, limitations of static LLM training cutoffs, workflow architecture diagrams, comparative analysis with Fine-Tuning and Prompt Engineering, and real-world industry adoption.

### Task 1: Understanding RAG & Daily App Improvement

#### What is Retrieval-Augmented Generation (RAG)?
Retrieval-Augmented Generation (RAG) is an AI architecture that bridges the gap between static Large Language Models (LLMs) and external, dynamic information sources. Instead of forcing an LLM to rely purely on the fixed weights it learned during training, RAG introduces a two-step mechanism:
1. **Retrieval:** When a user asks a question, the system queries an external knowledge store (such as a vector database, documents, live APIs, or enterprise files) to find the most relevant, up-to-date facts matching the query.
2. **Generation:** The retrieved facts are bundled alongside the original query into an augmented prompt. The LLM then reads this retrieved context and generates a precise, factually grounded answer with zero or minimal hallucinations.

In simple terms, standard LLMs act like a student taking a closed-book exam relying solely on memory, whereas RAG acts like an open-book exam where the student can look up the exact reference material before formulating their answer.

#### Real-World Example: Improving Flipkart Customer Experience
On e-commerce apps like **Flipkart**, users frequently ask highly specific technical and policy questions while viewing product pages, such as:
- *"Does this specific laptop model have an expandable RAM slot?"*
- *"Can I return this refurbished smartphone within 7 days if I do not like the display quality?"*

**Without RAG:** A standard LLM chatbot would give generic advice (e.g., "Most laptops support RAM upgrades") or hallucinate return policies that may violate seller terms.

**With RAG:** The chatbot retrieves the exact technical specification PDF, seller-specific return terms, and verified customer Q&A for that exact SKU from Flipkart's database. It then answers with 100% accuracy, citing exact details like "Yes, this model has 1 free SO-DIMM DDR5 slot supporting up to 32GB, and this specific seller offers a 7-day replacement policy."

### Task 2: Outdated LLM Knowledge & Real-World Consequences

#### 3 Questions a 2022-Trained Travel Assistant Bot Would Struggle to Answer:
1. **"What is the departure time and route schedule of the Vande Bharat Express between Ahmedabad and Mumbai Central?"**
   - *Why it fails:* Several Vande Bharat routes, timetables, and train numbers were introduced or modified in 2023 and 2024. A model trained in 2022 has no record of these trains.
2. **"Who won the IPL 2024 final, and who won the Orange Cap and Purple Cap awards in that season?"**
   - *Why it fails:* The IPL 2024 tournament occurred two full years past the 2022 training cutoff. The model has zero knowledge of the teams that reached the playoffs, the champion (Kolkata Knight Riders), or the award winners.
3. **"What is the current live running status and expected platform arrival for the Rajdhani Express at New Delhi Railway Station today?"**
   - *Why it fails:* Real-time railway operations change dynamically by the minute. Even if a model knew static 2022 schedules, it can never know live delays, track diversions, or platform assignments.

#### Why Outdated LLM Knowledge is a Critical Problem:
- **Hallucinations & False Confidence:** When an LLM lacks information about events after its cutoff date, it frequently invents convincing-sounding but completely fabricated answers rather than admitting ignorance.
- **Severe Real-World Impact:** In travel and ticketing domains (like IRCTC), acting on outdated schedules or incorrect platform numbers leads to passengers missing trains, monetary losses, safety risks, and immense frustration.
- **High Retraining Costs:** Foundation LLMs cannot simply be retrained every day or week whenever schedules or sports scores update; doing so would cost millions of dollars in compute. This makes external retrieval (RAG) an absolute necessity for real-time accuracy.

### Task 3: End-to-End RAG Architecture Workflow Diagram

The core RAG workflow operates across the following steps:
1. **User Query:** The user enters a question into the system.
2. **Embedding Model:** The text question is converted into a high-dimensional vector representation.
3. **Vector Database Retrieval:** The vector is matched against pre-indexed document embeddings using cosine similarity to retrieve the top-k most relevant chunks.
4. **Prompt Augmentation:** The user query and the retrieved context chunks are combined into an augmented prompt template.
5. **LLM Engine:** The LLM receives the enriched prompt, processes the contextual information, and reasons over the retrieved facts.
6. **Generated Answer:** The final, grounded response is delivered back to the user.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from IPython.display import Image, display

In [ ]:
def draw_rag_diagram():
    fig, ax = plt.subplots(figsize=(15, 8), dpi=300)
    ax.set_xlim(0, 15)
    ax.set_ylim(0, 8)
    ax.axis("off")

    def draw_box(x, y, w, h, title, subtitle, bg_color, border_color):
        box = patches.FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.2",
            facecolor=bg_color,
            edgecolor=border_color,
            linewidth=2.2
        )
        ax.add_patch(box)
        ax.text(x + w / 2, y + h * 0.62, title, ha="center", va="center", fontsize=12, fontweight="bold", color="#0f172a")
        if subtitle:
            ax.text(x + w / 2, y + h * 0.32, subtitle, ha="center", va="center", fontsize=9.5, color="#475569")

    draw_box(0.6, 5.0, 2.5, 1.5, "1. User Query", "Ask a question", "#e0f2fe", "#0284c7")
    draw_box(4.2, 5.0, 2.7, 1.5, "2. Embedding Model", "Convert query to vector", "#fef3c7", "#d97706")
    draw_box(4.2, 1.5, 2.7, 1.5, "3. Vector Database", "Indexed Documents & Embeddings", "#dcfce7", "#16a34a")
    draw_box(8.0, 5.0, 2.9, 1.5, "4. Prompt Augmentation", "Combine Query + Retrieved Chunks", "#f3e8ff", "#9333ea")
    draw_box(11.8, 5.0, 2.5, 1.5, "5. LLM Engine", "Reasoning & Synthesis", "#fee2e2", "#dc2626")
    draw_box(11.8, 1.5, 2.5, 1.5, "6. Generated Answer", "Accurate & Grounded Output", "#f1f5f9", "#334155")

    def draw_arrow(start, end, label="", curve=0.0):
        connectionstyle = f"arc3,rad={curve}" if curve != 0.0 else "arc3"
        ax.annotate(
            "",
            xy=end,
            xytext=start,
            arrowprops=dict(
                arrowstyle="->",
                lw=2.2,
                color="#334155",
                shrinkA=10,
                shrinkB=10,
                connectionstyle=connectionstyle
            )
        )
        if label:
            if curve == 0.0:
                mid_x = (start[0] + end[0]) / 2
                mid_y = (start[1] + end[1]) / 2 + 0.22
            else:
                mid_x = (start[0] + end[0]) / 2
                mid_y = max(start[1], end[1]) + abs(curve) * 2.0
            ax.text(mid_x, mid_y, label, ha="center", va="center", fontsize=9, color="#0f172a", fontweight="bold")

    draw_arrow((3.1, 5.75), (4.2, 5.75), "Text Query")
    draw_arrow((5.55, 5.0), (5.55, 3.0), "Vector Query")
    draw_arrow((6.9, 2.25), (8.5, 5.0), "Top-K Chunks")
    draw_arrow((1.85, 6.5), (9.45, 6.5), "Original User Query", curve=-0.22)
    draw_arrow((10.9, 5.75), (11.8, 5.75), "Context + Query")
    draw_arrow((13.05, 5.0), (13.05, 3.0), "Final Response")

    plt.title("Retrieval-Augmented Generation (RAG) End-to-End Workflow", fontsize=16, fontweight="bold", pad=25, color="#0f172a")
    plt.tight_layout()
    plt.savefig("rag_workflow_diagram.png", bbox_inches="tight")
    plt.close()

In [ ]:
draw_rag_diagram()

In [ ]:
display(Image(filename="rag_workflow_diagram.png"))

### Task 4: Comparison Table (RAG vs Fine-Tuning vs Prompt Engineering)

| Aspect / Dimension | Retrieval-Augmented Generation (RAG) | Fine-Tuning | Prompt Engineering |
| :--- | :--- | :--- | :--- |
| **Cost** | **Moderate:** Incurs expenses for vector database hosting, document chunk embedding generation, and runtime LLM inference tokens. No heavy GPU model training required. | **Highest:** Requires expensive GPU compute infrastructure for training runs, specialized dataset curation, continuous evaluation, and dedicated model hosting. | **Lowest:** Incurs zero training or vector database infrastructure costs; only standard runtime API token usage fees apply. |
| **Data Freshness** | **Dynamic / Real-time:** Instantly incorporates new data as soon as documents, database records, or files are updated or inserted into the vector store. | **Static / Delayed:** Frozen at the time of model checkpoint training; updating knowledge requires preparing new datasets and retraining the model. | **Static / Session-bound:** Limited to whatever static information can be manually pasted into the prompt context window for that single interaction. |
| **Use Case Example** | **Zomato / Swiggy Chatbot:** Pulling live restaurant menus, current discounts, active operating hours, and order tracking from production databases. | **BloombergGPT / Medical Diagnostic Models:** Training on proprietary financial dialect or medical terminologies and clinical note formats. | **Spotify Playlist Curator Persona:** Instructing an off-the-shelf LLM to speak in a hipster tone and recommend songs formatted in strict JSON. |
| **Speed of Implementation** | **Fast to Moderate:** Takes hours to a few days to chunk documents, generate embeddings, index in Chroma/Pinecone, and link a retrieval chain. | **Slow:** Takes days to weeks or months to collect high-quality training pairs, clean data, run training epochs, evaluate against benchmarks, and deploy. | **Immediate:** Takes minutes to hours; developers simply refine prompt instructions, few-shot examples, and system messages in code. |

#### Strategic Summary
- Use **Prompt Engineering** when you need quick prototypes, persona styling, or formatting tasks.
- Use **RAG** when your application needs dynamic, up-to-date facts, private enterprise knowledge, or verifiable source citations.
- Use **Fine-Tuning** when you need to teach a model a specialized vocabulary, complex internal syntax, or consistent structural outputs where knowledge retrieval alone is insufficient.

### Task 5: Real-World Industry Application of RAG

#### Industry Example: Notion AI (Workspace Q&A)
A leading real-world application of RAG is **Notion AI's Q&A** feature, which allows teams and enterprises to ask natural language questions across thousands of internal workspace pages, project roadmaps, and meeting notes.

#### How RAG Powers Notion AI:
1. **Continuous Vector Indexing:** Whenever a team member creates or updates a document, page, or table in Notion, the platform chunks the content, computes vector embeddings, and stores them in a secure vector index.
2. **Permission-Aware Retrieval:** When an employee asks a question like *"What is our company's maternity leave policy in the UK office?"*, Notion embeds the question and retrieves only the matching document chunks that the employee has access rights to view.
3. **Context Injection & Answer Generation:** The retrieved policy snippets are fed to the LLM alongside the question, instructing it to synthesize a concise summary exclusively using those snippets.
4. **Exact Citations:** Notion AI renders clickable citations below the answer pointing to the exact internal pages where the information originated.

#### Why RAG Makes Notion AI Superior:
- **Eliminates Hallucination on Private Data:** Standard foundation models have never seen a private company's internal HR policies or roadmap. RAG grounds every response in the actual text written by the team.
- **Enterprise Security & Access Control:** RAG enables filtering retrieval results strictly by user permissions, preventing unauthorized staff from seeing confidential executive documents.
- **Instant Verification:** Employees can click directly on the cited document links to verify the policy details firsthand.